# 05 - Fluxo de Dados

## Descrição

Este notebook documenta o fluxo completo de dados no sistema, desde a origem MongoDB ate o consumo analitico na camada Gold. Inclui diagramas de sequencia detalhados para cada DAG, mapeamento de transformacoes, e analise de como os dados se movem, sao transformados e persistidos em cada etapa.

## Fontes de Dados

### Fonte Primaria: MongoDB

| Caracteristica | Descricao |
|---------------|-----------|
| **Banco** | ecommerce |
| **Colecoes** | 10 (clientes, categorias, fornecedores, produtos, cupons, pedidos, itens_pedido, pagamentos, entregas, avaliacoes) |
| **Volume** | ~15.000 documentos por colecao (total ~150.000) |
| **Formato** | BSON (JSON binario com tipos nativos: ISODate, NumberInt, NumberDouble, ObjectId) |
| **Chaves** | IDs inteiros sequenciais (1..15000), nao ObjectId |
| **Relacionamentos** | Referenciados por chaves inteiras (FK sorteadas no intervalo 1..15000) |
| **Incremental** | Campo updated_at (date) presente em todas as colecoes |
| **Deploy** | Local (Docker: localhost:27017) ou Atlas (mongodb+srv://) |

### Dados Sinteticos (Dataset)

| Caracteristica | Descricao |
|---------------|-----------|
| **Gerador** | Faker (locale pt_BR) + scripts Python customizados |
| **Sementes** | Fixas (deterministicas) - random.seed(), Faker.seed() |
| **Formato** | CSV (intermediario) -> BSON (MongoDB) |
| **Localizacao** | dataset/arquivos_csv/ (nao versionados, ~15MB) |

---

## Linhagem de Dados (Data Lineage)

```mermaid
flowchart LR
    subgraph Origem
        M[MongoDB\nBSON\n10 colecoes]
    end

    subgraph Landing
        L[JSON Lines\nExtended JSON\nTipos BSON preservados]
    end

    subgraph Bronze
        B[Delta Lake\nSchema: raw_document + metadata\nParticao: ingestion_date]
    end

    subgraph Silver
        S[Delta Lake\nSchema tipado + controle\nDeduplicado + validado]
    end

    subgraph Gold
        G[Delta Lake\nDimensoes SCD2 + Fatos\nParticao: ano (fatos)]
    end

    subgraph Consumo
        D[Dashboard/BI\nSQL/OLAP\nKPIs e metricas]
    end

    M -->|DAG 1: mongodb_to_landing<br/>Extracao incremental<br/>Checkpoint: updated_at| L
    L -->|DAG 2: landing_to_bronze<br/>Conversao JSON -> Delta<br/>Metadados de auditoria| B
    B -->|DAG 3: bronze_to_silver<br/>Limpeza, deduplicacao<br/>Validacao, MERGE| S
    S -->|DAG 4: silver_to_gold<br/>Modelagem dimensional<br/>SCD2, point-in-time| G
    G -->|Consultas SQL<br/>Delta Lake connector| D

    style M fill:#15803d,color:#fff,stroke:#166534
    style L fill:#e2e8f0,stroke:#94a3b8,color:#1e293b
    style B fill:#fdba74,stroke:#b45309,color:#7c2d12
    style S fill:#cbd5e1,stroke:#64748b,color:#1e293b
    style G fill:#fde047,stroke:#a16207,color:#713f12
    style D fill:#ede9fe,stroke:#7c3aed,color:#4c1d95
```

---

## Transformacoes por Camada

### Landing (Extracao)

| Transformacao | Descricao | Exemplo |
|--------------|-----------|---------|
| **Serializacao** | BSON -> MongoDB Extended JSON | ISODate(2024-01-15) -> {\$date: {\$numberLong: ...}} |
| **Particionamento** | Organizacao por data e execucao | extraction_date=2026-06-13/run_id=manual__.../part-00000.json |
| **Incremental** | Filtro por checkpoint | updated_at >= checkpoint - 24h |
| **Streaming** | Escrita linha a linha (nao carrega tudo em memoria) | for doc in cursor: write_json_line(doc) |

### Bronze (Conversao)

| Transformacao | Descricao | Exemplo |
|--------------|-----------|---------|
| **Parsing JSON** | Texto -> Estrutura Delta | spark.read.json(...) |
| **Metadata Extraction** | Extrai info do path do arquivo | _bronze_extraction_date do path extraction_date=2026-06-13 |
| **Auditoria** | Adiciona colunas de controle | _bronze_source_file, _bronze_airflow_run_id, _bronze_ingested_at |
| **Idempotencia** | Evita reprocessar arquivos ja ingeridos | left_anti join em _bronze_source_file |
| **Particionamento** | Particao por data de ingestao | partitionBy(ingestion_date) |

### Silver (Limpeza e Conformacao)

| Transformacao | Descricao | Exemplo |
|--------------|-----------|---------|
| **Type Casting** | BSON types -> Spark types | NumberInt -> long, ISODate -> timestamp |
| **Normalization** | Trim, lower, upper, digits-only | cpf -> apenas digitos (remove . e -) |
| **Deduplication** | Mantem mais recente por PK | row_number() over (partition by id order by updated_at desc) |
| **Quality Validation** | Required, enum, regex, range | estado pertence {AC, AL, ..., SP}, email match ^[^@]+@[^@]+\.[^@]+$ |
| **Referential Integrity** | FKs validas contra tabelas pai | pedidos.id_cliente existe em clientes.id_cliente |
| **Business Rules** | Unicidade de negocio | CPF unico, um pagamento aprovado por pedido |
| **MERGE** | Sincronizacao incremental | insert novos, update alterados, ignore inalterados |

### Gold (Modelagem Dimensional)

| Transformacao | Descricao | Exemplo |
|--------------|-----------|---------|
| **Dimension Building** | Construcao de dimensoes | dim_cliente a partir de clientes Silver |
| **SCD Type 2** | Versionamento de atributos | dw_valid_from, dw_valid_to, dw_is_current |
| **Surrogate Key** | Chave artificial deterministica | SHA256(cliente_key + dw_valid_from) |
| **Fact Building** | Construcao de fatos com joins | fato_vendas = itens_pedido JOIN pedidos |
| **Point-in-Time Join** | Anexa SK vigente na data do evento | cliente_sk vigente em data_pedido |
| **Metrics Calculation** | Metricas de negocio | valor_bruto = quantidade * valor_unitario, receita_liquida = subtotal |
| **Partitioning** | Particao por ano para performance | partitionBy(ano) nos fatos |
| **MERGE** | Sincronizacao com delete | insert + update + delete (registros removidos da Silver) |

---

## Diagramas de Sequencia Detalhados por DAG

### DAG 1: mongodb_to_landing (Extracao Incremental)

```mermaid
sequenceDiagram
    participant Scheduler as Airflow Scheduler
    participant DAG as DAG: mongodb_to_landing
    participant Task1 as Task: validate_landing_bucket
    participant Task2 as Task: validate_source_collections
    participant Task3 as Task: extract_collection (expand)
    participant Task4 as Task: write_run_manifest
    participant Mongo as MongoDB Atlas
    participant Var as Airflow Variables
    participant S3 as MinIO/S3

    Scheduler->>DAG: Trigger (schedule */15 * * * *)
    DAG->>Task1: Execute
    Task1->>S3: head_bucket(datalake)
    S3-->>Task1: OK (200) / Not Found (404)
    Task1-->>DAG: Bucket status

    DAG->>Task2: Execute
    Task2->>Mongo: list_collection_names(ecommerce)
    Mongo-->>Task2: [clientes, categorias, ...]
    Task2->>Task2: Compare with EXPECTED_COLLECTIONS
    Task2-->>DAG: Collections list / Exception

    DAG->>Task3: Execute (parallel for each collection)
    loop Para cada colecao (10x em paralelo)
        Task3->>Var: get(checkpoint_variable_name)
        Var-->>Task3: checkpoint (ISO-8601) / null
        Task3->>Task3: build_incremental_filter(checkpoint, updated_at, 24h)
        Task3->>Mongo: find(filter).sort(updated_at).batch_size(1000)
        Mongo-->>Task3: Cursor
        Task3->>Task3: Stream to JSON Lines (write_json_lines)
        Task3->>Task3: Track max_updated_at
        alt Documentos encontrados
            Task3->>S3: PUT part-00000.json
            Task3->>Var: set(checkpoint_variable, max_updated_at)
        else Sem documentos
            Task3->>Task3: Nao grava arquivo, mantem checkpoint
        end
        Task3-->>DAG: Result dict (collection, count, object_key, checkpoint)
    end

    DAG->>Task4: Execute
    Task4->>Task4: build_manifest(results)
    Task4->>S3: PUT manifest.json
    Task4-->>DAG: Manifest key
    DAG-->>Scheduler: Success / Failure
```

### DAG 2: landing_to_bronze (Conversao JSON -> Delta)

```mermaid
sequenceDiagram
    participant Scheduler as Airflow Scheduler
    participant DAG as DAG: landing_to_bronze
    participant Task1 as Task: validate_data_lake
    participant Task2 as Task: convert_landing_to_bronze (SparkSubmitOperator)
    participant S3 as MinIO/S3
    participant Spark as Spark Job

    Scheduler->>DAG: Trigger (schedule 5-59/15 * * * *)
    DAG->>Task1: Execute
    Task1->>S3: head_bucket(datalake)
    S3-->>Task1: OK
    Task1->>S3: head_object(bronze/ecommerce/<col>/_READY) for each collection
    S3-->>Task1: OK
    Task1->>S3: list_objects(landing/ecommerce/<col>/*.json) for each collection
    S3-->>Task1: File list
    Task1-->>DAG: Validation result (counts)

    DAG->>Task2: Execute
    Task2->>Spark: spark-submit landing_to_bronze.py
    Spark->>Spark: Create SparkSession with Delta + S3A config
    loop Para cada colecao
        Spark->>S3: READ JSON Lines (recursive, pathGlobFilter=*.json)
        S3-->>Spark: Raw JSON text
        Spark->>Spark: Extract metadata from path (extraction_date, run_id via regex)
        Spark->>Spark: Add audit columns (_bronze_source_file, _bronze_ingested_at, etc.)
        alt Delta table exists
            Spark->>Spark: Read existing _bronze_source_file
            Spark->>Spark: left_anti join to filter already processed files
        end
        Spark->>S3: WRITE Delta Lake (append, partitionBy=ingestion_date, mergeSchema=true)
    end
    Spark->>S3: PUT manifest.json
    Spark-->>Task2: Exit code 0
    Task2-->>DAG: Success
    DAG-->>Scheduler: Success / Failure
```

### DAG 3: bronze_to_silver (Limpeza e Validacao)

```mermaid
sequenceDiagram
    participant Scheduler as Airflow Scheduler
    participant DAG as DAG: bronze_to_silver
    participant Task1 as Task: validate_data_lake
    participant Task2 as Task: clean_bronze_to_silver (SparkSubmitOperator)
    participant S3 as MinIO/S3
    participant Spark as Spark Job

    Scheduler->>DAG: Trigger (schedule 10-59/15 * * * *)
    DAG->>Task1: Execute
    Task1->>S3: head_bucket(datalake)
    S3-->>Task1: OK
    Task1->>S3: head_object(silver/ecommerce/<table>/_READY) for each table
    S3-->>Task1: OK
    Task1->>S3: list_objects(bronze/ecommerce/<table>/_delta_log/) for each table
    S3-->>Task1: Delta log exists
    Task1-->>DAG: Validation result

    DAG->>Task2: Execute
    Task2->>Spark: spark-submit bronze_to_silver.py
    Spark->>Spark: Create SparkSession with Delta + S3A config
    loop Para cada tabela (em ordem de dependencia)
        Spark->>S3: READ Delta Bronze (S3A)
        S3-->>Spark: Delta table data
        Spark->>Spark: Convert BSON types (NumberInt -> long, ISODate -> timestamp)
        Spark->>Spark: Normalize strings (trim, lower, digits-only)
        Spark->>Spark: Deduplicate by PK (window: updated_at DESC, _bronze_ingested_at DESC)
        Spark->>Spark: Validate quality (required, enum, regex, range per ENTITY_RULES)
        Spark->>Spark: Apply business unique rules (CPF, approved payment)
        Spark->>S3: WRITE rejected records to quality_log (append-only Delta)
        Spark->>Spark: Validate foreign keys (broadcast join with parent Silver tables)
        alt Silver table exists
            Spark->>Spark: MERGE (insert new, update changed, ignore unchanged)
            Spark->>S3: WRITE Delta Silver (MERGE)
        else New table
            Spark->>S3: WRITE Delta Silver (append mode)
        end
    end
    Spark->>S3: PUT manifest.json
    Spark-->>Task2: Exit code 0
    Task2-->>DAG: Success
    DAG-->>Scheduler: Success / Failure
```

### DAG 4: silver_to_gold (Modelagem Dimensional)

```mermaid
sequenceDiagram
    participant Scheduler as Airflow Scheduler
    participant DAG as DAG: silver_to_gold
    participant Task1 as Task: validate_data_lake
    participant Task2 as Task: build_silver_to_gold (SparkSubmitOperator)
    participant S3 as MinIO/S3
    participant Spark as Spark Job

    Scheduler->>DAG: Trigger (schedule 15-59/15 * * * *)
    DAG->>Task1: Execute
    Task1->>S3: head_bucket(datalake)
    S3-->>Task1: OK
    Task1->>S3: head_object(gold/ecommerce/<model>/_READY) for each model
    S3-->>Task1: OK
    Task1->>S3: list_objects(silver/ecommerce/<table>/_delta_log/) for required tables
    S3-->>Task1: Delta log exists
    Task1-->>DAG: Validation result

    DAG->>Task2: Execute
    Task2->>Spark: spark-submit silver_to_gold.py
    Spark->>Spark: Create SparkSession with Delta + S3A config
    Spark->>S3: READ Delta Silver (S3A) - all required tables
    S3-->>Spark: Delta table data

    Note over Spark: Construcao de Dimensoes
    Spark->>Spark: Build dim_tempo (explode sequence of dates)
    Spark->>Spark: Build dim_cliente (SCD2: hash, valid_from, valid_to, is_current)
    Spark->>Spark: Build dim_produto (SCD2: join categorias + fornecedores)
    Spark->>Spark: Build dim_cupom (SCD2)
    Spark->>S3: MERGE dimensoes SCD2 (expira + insere)

    Note over Spark: Construcao de Fatos
    Spark->>Spark: Build fato_vendas (join itens_pedido + pedidos)
    Spark->>Spark: Build fato_pagamentos (join pagamentos + pedidos)
    Spark->>Spark: Build fato_entregas (join entregas + pedidos)
    Spark->>Spark: Build fato_avaliacoes (join avaliacoes)
    Spark->>Spark: Point-in-time join: anexa SKs vigentes
    Spark->>Spark: Calcula metricas (receita, atraso, aprovacao, etc.)
    Spark->>S3: MERGE fatos (insert + update + delete)

    Spark->>S3: PUT manifest.json
    Spark-->>Task2: Exit code 0
    Task2-->>DAG: Success
    DAG-->>Scheduler: Success / Failure
```

---

## Comunicacoes e APIs

### Airflow -> MongoDB (Hook)

- **Hook**: MongoHook (provider apache-airflow-providers-mongo)
- **Connection**: mongodb_atlas (URI no formato MongoDB)
- **Operacao**: find(filter).sort().batch_size(1000)
- **Retorno**: Cursor iteravel de documentos BSON

### Airflow -> S3/MinIO (Hook)

- **Hook**: S3Hook (provider apache-airflow-providers-amazon)
- **Connection**: minio_s3 (endpoint, key, secret, region)
- **Operacoes**: check_for_bucket(), load_file(), load_string(), list_keys()
- **Retorno**: Status booleano, chaves de objetos

### Airflow -> Spark (Operator)

- **Operator**: SparkSubmitOperator (provider apache-airflow-providers-apache-spark)
- **Connection**: spark_default (master URL, deploy mode)
- **Argumentos**: application (path PySpark), packages (Delta + Hadoop AWS), conf (S3A config), application_args (CLI args)
- **Retorno**: Exit code do Spark job (0 = success)

### Spark -> S3/MinIO (S3A FileSystem)

- **Protocolo**: S3A (Hadoop S3A FileSystem)
- **Configuracao**: spark.hadoop.fs.s3a.endpoint, spark.hadoop.fs.s3a.path.style.access=true
- **Credenciais**: AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY (env vars)
- **Formato**: Delta Lake (transacoes ACID sobre S3A)

---

## Mecanismos de Persistencia

### Object Storage (MinIO / S3)

| Camada | Formato | Estrutura | Particao |
|--------|---------|-----------|----------|
| Landing | JSON Lines | landing/ecommerce/<col>/extraction_date=<date>/run_id=<id>/part-00000.json | extraction_date |
| Bronze | Delta Lake | bronze/ecommerce/<col>/_delta_log/, .../ingestion_date=<date>/*.parquet | ingestion_date |
| Silver | Delta Lake | silver/ecommerce/<col>/_delta_log/, .../*.parquet | Nenhuma |
| Gold | Delta Lake | gold/ecommerce/<model>/_delta_log/, .../ano=<year>/*.parquet (fatos) | ano (fatos) |

### Versionamento (S3 Versioning)

- Habilitado por padrao nos scripts de estrutura (put_bucket_versioning: Enabled)
- Permite recuperacao de versoes anteriores de manifestos e dados
- Nao substitui o time-travel do Delta Lake, mas complementa para objetos nao-Delta

### Cache e Memoria

- Spark usa MEMORY_AND_DISK para DataFrames intermediarios (ex: candidates.persist() na Silver)
- Unpersist explicito apos uso para liberar memoria
- MongoDB cursor com batch_size=1000 para nao carregar tudo em memoria
